# K-Means Project Tutorial: House grouping system

Classify California census-block groups by **region** and **median income** using `MedInc`, `Latitude`, and `Longitude`.

The train/test split is not used for supervised metrics here. We fit K-Means on `train`, then assign each unseen `test` house to the cluster it belongs to.

## Step 1: Loading the dataset

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/raw/housing.csv")

total_data = pd.read_csv(DATA_PATH)
print(f"Shape: {total_data.shape}")
print(f"Missing values: {total_data.isnull().sum().sum()}")
total_data.head()

### Keep only the clustering features

In [ ]:
features = ["MedInc", "Latitude", "Longitude"]
X = total_data[features].copy()

print(X.describe())
X.head()

### Split into train and test

K-Means will learn the cluster centers from `X_train`. `X_test` is held out so we can later predict which cluster new houses belong to.

In [ ]:
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
X_train = X_train.copy()
X_test = X_test.copy()

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
X_train.head()

## Step 2: Build a K-Means

Classify the training houses into **6 clusters**. Store the assigned group as a `cluster` column, inspect its format, and convert it to a category if the labels are discrete group IDs rather than a numeric scale.

In [ ]:
from sklearn.cluster import KMeans

model_unsup = KMeans(n_clusters=6, n_init="auto", random_state=42)
model_unsup.fit(X_train[features])

X_train["cluster"] = model_unsup.labels_

print("cluster dtype:", X_train["cluster"].dtype)
print("unique values:", sorted(X_train["cluster"].unique()))
print(X_train["cluster"].value_counts().sort_index())

# Labels are integers 0-5. They name groups, not a numeric scale.
X_train["cluster"] = X_train["cluster"].astype("category")
print("\ncluster dtype after categorizing:", X_train["cluster"].dtype)
X_train.head()

### Plot the training clusters

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axis = plt.subplots(1, 3, figsize=(15, 5))

sns.scatterplot(ax=axis[0], data=X_train, x="Longitude", y="Latitude", hue="cluster", palette="deep")
sns.scatterplot(ax=axis[1], data=X_train, x="Latitude", y="MedInc", hue="cluster", palette="deep")
sns.scatterplot(ax=axis[2], data=X_train, x="Longitude", y="MedInc", hue="cluster", palette="deep")

axis[0].set_title("Geographic clusters")
axis[1].set_title("Income vs latitude")
axis[2].set_title("Income vs longitude")
plt.tight_layout()
plt.show()

In [ ]:
cluster_profile = (
    X_train.groupby("cluster", observed=True)[["MedInc", "Latitude", "Longitude"]]
    .mean()
    .round(2)
)
cluster_profile["houses"] = X_train["cluster"].value_counts().sort_index()
cluster_profile

`cluster` starts as integers `0`–`5`. Those values only name groups, so the column is stored as a category before plotting.

The Longitude vs Latitude scatter splits California into two regions, then income splits each region:

- **Northern California** (higher latitude, more negative longitude): cluster **0** is upper-middle income (mean MedInc ≈ 5.4, including much of the Bay Area) and cluster **5** is a large lower-income northern group (mean ≈ 2.7).
- **Southern California** (Los Angeles, Orange County, San Diego): cluster **2** is affluent (mean ≈ 6.9), cluster **1** is middle income (mean ≈ 4.4), and cluster **3** is the large lower-income southern group (mean ≈ 2.4).
- **Cluster 4** is a small set of very high-income houses (n = 246, mean MedInc ≈ 11.8). It is not one city; those points sit in wealthy coastal pockets.

K-Means is grouping houses by **where they are** and **how wealthy the block is**, which matches the goal of classifying houses by region and median income.

## Step 3: Predict with the test set

The test houses were never seen during `fit`. Use the trained model to predict the cluster each one belongs to, then add those points to the plot above to confirm whether the prediction is successful.

In [ ]:
X_test["cluster"] = model_unsup.predict(X_test[features])
X_test["cluster"] = X_test["cluster"].astype("category")

print(X_test["cluster"].value_counts().sort_index())
X_test.head()

In [ ]:
comparison = pd.DataFrame({
    "train_%": (X_train["cluster"].value_counts(normalize=True).sort_index() * 100).round(2),
    "test_%": (X_test["cluster"].value_counts(normalize=True).sort_index() * 100).round(2),
    "test_houses": X_test["cluster"].value_counts().sort_index(),
    "train_MedInc": X_train.groupby("cluster", observed=True)["MedInc"].mean().round(2),
    "test_MedInc": X_test.groupby("cluster", observed=True)["MedInc"].mean().round(2),
})
comparison

### Overlay test predictions on the training plot

In [ ]:
centers = pd.DataFrame(model_unsup.cluster_centers_, columns=features)

fig, axis = plt.subplots(1, 3, figsize=(15, 5))

pairs = [("Longitude", "Latitude"), ("Latitude", "MedInc"), ("Longitude", "MedInc")]

for ax, (x_col, y_col) in zip(axis, pairs):
    # Training houses: faded background
    sns.scatterplot(ax=ax, data=X_train, x=x_col, y=y_col, hue="cluster", palette="deep", alpha=0.25, s=12, linewidth=0, legend=(ax is axis[0]))
    # Test predictions: "+" markers on top
    sns.scatterplot(ax=ax, data=X_test, x=x_col, y=y_col, hue="cluster", palette="deep", marker="+", s=22, linewidth=0.7, legend=False)
    # Cluster centers learned from train
    ax.scatter(centers[x_col], centers[y_col], c="black", marker="X", s=160, zorder=5)
    ax.set_title(f"{y_col} vs {x_col}")

axis[0].set_title("Test predictions over training clusters")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Distance from each house to the center of the cluster it was assigned to.
train_dist = np.linalg.norm(X_train[features].values - model_unsup.cluster_centers_[X_train["cluster"].astype(int)], axis=1)
test_dist = np.linalg.norm(X_test[features].values - model_unsup.cluster_centers_[X_test["cluster"].astype(int)], axis=1)

print(f"Mean distance to assigned center - train: {train_dist.mean():.3f} | test: {test_dist.mean():.3f}")

# Every test house should be closest to the center it was assigned to.
nearest = np.argmin(np.linalg.norm(X_test[features].values[:, None, :] - model_unsup.cluster_centers_[None, :, :], axis=2), axis=1)
print(f"Test houses assigned to their nearest center: {(nearest == X_test['cluster'].astype(int)).mean():.1%}")

**The prediction is successful.** The `+` markers (test houses) fall inside the same colored regions as the faded training points, with no stray markers landing in a foreign group. The black `X` markers are the centers learned from `train`, and the test points sit around those same centers.

The numbers confirm what the plot shows:

- **Cluster sizes match.** Each cluster holds nearly the same share of houses in both sets (for example cluster 3: 26.45% of train vs 26.91% of test; cluster 4: 1.49% vs 1.50%).
- **Cluster meaning is stable.** Mean income per cluster is almost identical across sets (cluster 2: 6.94 train vs 6.96 test; cluster 4: 11.75 vs 11.72).
- **Test points are not worse-fitted.** Mean distance to the assigned center is 1.217 for train and 1.200 for test, so unseen houses sit just as close to their group as the houses used for fitting.
- **Every test house was assigned to its nearest center** (100% agreement), which is exactly what `predict` should do.

Since the held-out houses land in the same regional and income groups as the training data, the model generalizes to new points rather than memorizing the training set.

## Step 4: Train a supervised classification model

K-Means labeled every house for us, so we now have a **target**: the `cluster` column. That turns the problem into supervised multi-class classification — predict the cluster from `MedInc`, `Latitude`, and `Longitude`.

Which model is most useful? K-Means assigns each point to its nearest center, so the true boundaries are straight lines between centers (a Voronoi partition). Models that can carve the feature space into regions fit this well:

- **Decision tree** — axis-aligned splits, fast and easy to read.
- **Random forest** — averages many trees, smoother boundaries.
- **K-nearest neighbors** — naturally matches "closest group" logic.
- **Logistic regression** — draws linear boundaries, which is the right shape here, but needs scaling since `MedInc` and the coordinates have different ranges.

Train the candidates and compare before committing to one.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

y_train = X_train["cluster"]
y_test = X_test["cluster"]

candidates = {
    "Decision tree (depth 8)": DecisionTreeClassifier(max_depth=8, random_state=42),
    "Random forest (200)": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
}

scores = []
for name, candidate in candidates.items():
    candidate.fit(X_train[features], y_train)
    scores.append({
        "model": name,
        "train_accuracy": round(candidate.score(X_train[features], y_train), 4),
        "test_accuracy": round(candidate.score(X_test[features], y_test), 4),
        "cv5_accuracy": round(cross_val_score(candidate, X_train[features], y_train, cv=5).mean(), 4),
    })

pd.DataFrame(scores).set_index("model").sort_values("test_accuracy", ascending=False)

### Train the chosen model

Every candidate clears 99%, which is expected because the labels come from a geometric rule rather than noisy real-world outcomes. The **random forest** has the best cross-validated and test accuracy, so use it as the final classifier.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model_sup = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model_sup.fit(X_train[features], y_train)

y_pred = model_sup.predict(X_test[features])

print(f"Train accuracy: {model_sup.score(X_train[features], y_train):.4f}")
print(f"Test accuracy:  {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(
    confusion_matrix(y_test, y_pred),
    annot=True, fmt="d", cmap="Blues", cbar=False, ax=axis[0],
)
axis[0].set_xlabel("Predicted cluster")
axis[0].set_ylabel("Actual cluster")
axis[0].set_title("Confusion matrix (test set)")

importances = pd.Series(model_sup.feature_importances_, index=features).sort_values()
importances.plot.barh(ax=axis[1], color="steelblue")
axis[1].set_title("Feature importance")
axis[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()

In [ ]:
# Where do the mistakes happen? Compare the distance to the nearest center against the
# second-nearest one. A small gap means the house sits on the border between two clusters.
distances = np.linalg.norm(X_test[features].values[:, None, :] - model_unsup.cluster_centers_[None, :, :], axis=2)
nearest_two = np.sort(distances, axis=1)[:, :2]
margin = nearest_two[:, 1] - nearest_two[:, 0]

wrong = y_pred != y_test
print(f"Misclassified houses: {wrong.sum()} of {len(y_test)}")
print(f"Median border margin - correct: {np.median(margin[~wrong]):.3f} | misclassified: {np.median(margin[wrong]):.3f}")

### What the statistics show

The random forest reaches **99.64% test accuracy** (15 wrong out of 4,128 houses), with 5-fold cross-validation at 99.41%, so the score is not an artifact of one lucky split.

**Per-class behavior.** Precision and recall are above 0.99 for every cluster except one. Cluster 4 — the small, very high-income group with only 62 test houses — has perfect precision (1.000) but lower recall (0.968): the model never labels another house as cluster 4 by mistake, yet it misses 2 of them, sending both to cluster 2 (the next-richest group). Small classes are simply harder, since there is less to learn from.

**The confusion matrix is nearly diagonal.** All 15 errors fall between clusters that are neighbors in income, in location, or both — the largest pair is 0↔5, the two northern clusters that differ mainly by income (5 houses), followed by 1↔3 in the south (3 houses) and 4→2 at the top of the income range (2 houses).

**The errors sit on cluster borders.** For correctly classified houses, the gap between the nearest and second-nearest center is 1.205 (median). For misclassified houses it is 0.024 — about fifty times smaller. Those houses are almost exactly equidistant from two centers, so either label is defensible.

**Feature importance.** `MedInc` accounts for roughly 58% of the splits, with `Latitude` (22%) and `Longitude` (20%) sharing the rest. Income is the single strongest signal, but geography is what separates clusters that share an income level.

**Why accuracy is so high.** The labels came from K-Means, which assigns each house to its nearest center using these same three columns. The target is a clean geometric rule, not a noisy real-world outcome, so a flexible classifier can recover it almost perfectly. The payoff is practical: the classifier now reproduces the grouping on its own, so new houses can be labeled without re-running K-Means.

## Step 5: Save the models

Store both models in `../models/`: the unsupervised K-Means that created the groups, and the supervised random forest that reproduces them. The labeled train/test sets go to `../data/processed/`.

In [ ]:
from pathlib import Path
from pickle import dump

processed_dir = Path("../data/processed")
models_dir = Path("../models")
processed_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(processed_dir / "housing_train.csv", index=False)
X_test.to_csv(processed_dir / "housing_test.csv", index=False)

# Unsupervised model: creates the groups
with open(models_dir / "kmeans_housing.sav", "wb") as file:
    dump(model_unsup, file)

# Supervised model: reproduces the groups for new houses
with open(models_dir / "random_forest_housing.sav", "wb") as file:
    dump(model_sup, file)

for saved in sorted(models_dir.glob("*.sav")):
    print(f"{saved.name} ({saved.stat().st_size / 1024:.0f} KB)")

In [ ]:
from pickle import load

with open(models_dir / "kmeans_housing.sav", "rb") as file:
    reloaded_kmeans = load(file)
with open(models_dir / "random_forest_housing.sav", "rb") as file:
    reloaded_forest = load(file)

# A house in the Bay Area with a median income of 8.3.
new_house = pd.DataFrame([[8.3252, 37.88, -122.23]], columns=features)

print("K-Means cluster:     ", reloaded_kmeans.predict(new_house)[0])
print("Random forest cluster:", reloaded_forest.predict(new_house)[0])